# Generación de Texto con modelos GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arlexpin/icesi-nlp/blob/main/Entregables/Sesion5/Work5.ipynb)

En este notebook harémos uso de un modelo tipo GPT-2 pre-entrenado en idioma español que utilizaremos para generar texto a partir de un contexto inicial. Luego, harémos fine tuning a este modelo con un dataset de recetas de cocina y observar como cambia la generación de texto en función del dataset que utilicemos.

#### Referencias
- Dataset: https://huggingface.co/datasets/Frorozcol/recetas-cocina
- [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [GPT2 Spanish](https://huggingface.co/mrm8488/spanish-gpt2)
- [Fine-Tune a non-Englush GPT-2 Model with Huggingface](https://www.philschmid.de/fine-tune-a-non-english-gpt-2-model-with-huggingface)

In [1]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

C:\Users\apapa\AppData\Local\Temp\ipykernel_50284\2396000874.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [2]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]' sentence-transformers torchinfo evaluate

"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## Generative pre-training Transformer - GPT

In [3]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "mrm8488/spanish-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  

model.config.pad_token_id = tokenizer.pad_token_id

model.resize_token_embeddings(len(tokenizer))

model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: mrm8488/spanish-gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50266, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
modules = [m for m, _ in model.named_modules()]
modules

['',
 'transformer',
 'transformer.wte',
 'transformer.wpe',
 'transformer.drop',
 'transformer.h',
 'transformer.h.0',
 'transformer.h.0.ln_1',
 'transformer.h.0.attn',
 'transformer.h.0.attn.c_attn',
 'transformer.h.0.attn.c_proj',
 'transformer.h.0.attn.attn_dropout',
 'transformer.h.0.attn.resid_dropout',
 'transformer.h.0.ln_2',
 'transformer.h.0.mlp',
 'transformer.h.0.mlp.c_fc',
 'transformer.h.0.mlp.c_proj',
 'transformer.h.0.mlp.act',
 'transformer.h.0.mlp.dropout',
 'transformer.h.1',
 'transformer.h.1.ln_1',
 'transformer.h.1.attn',
 'transformer.h.1.attn.c_attn',
 'transformer.h.1.attn.c_proj',
 'transformer.h.1.attn.attn_dropout',
 'transformer.h.1.attn.resid_dropout',
 'transformer.h.1.ln_2',
 'transformer.h.1.mlp',
 'transformer.h.1.mlp.c_fc',
 'transformer.h.1.mlp.c_proj',
 'transformer.h.1.mlp.act',
 'transformer.h.1.mlp.dropout',
 'transformer.h.2',
 'transformer.h.2.ln_1',
 'transformer.h.2.attn',
 'transformer.h.2.attn.c_attn',
 'transformer.h.2.attn.c_proj',
 'tran

Observermos un ejemplo de generación simple.

In [5]:
text = "Quiero algo rico con pollo"
best = 10

with torch.no_grad():
    tokens = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    print("Dimensiones de la entrada:", tokens.shape)
    output = model(input_ids=tokens)
    print("Dimensiones de la salida:", output.logits.shape)
    output = output.logits[0, -1, :]
    print("Dimensiones del último token de la secuencia:", output.shape)
    probs = torch.softmax(output, dim=-1)
    print("Dimensiones de la probabilidad de los tokens:", probs.shape)
    sorted_probs = torch.argsort(probs, dim=-1, descending=True)
    print({tokenizer.decode(token): f"{prob.cpu().numpy() * 100:.2f}%" for token, prob in zip(sorted_probs[:best], probs[sorted_probs[:best]])})

Dimensiones de la entrada: torch.Size([1, 5])
Dimensiones de la salida: torch.Size([1, 5, 50266])
Dimensiones del último token de la secuencia: torch.Size([50266])
Dimensiones de la probabilidad de los tokens: torch.Size([50266])
{'.': '38.99%', ' y': '17.26%', ',': '11.24%', ' frito': '4.53%', '...': '3.22%', ' para': '3.00%', ' a': '1.90%', ' en': '1.57%', ' con': '0.89%', '!': '0.80%'}


## Implementando una función de generación

Ahora, la idea es que este modelo nos sirva para generar texto de forma recurrente e incremental. En la última capa de los modelos tipo GPT encontrarémos un tensor con forma $(b, s, v)

In [6]:
import torch.nn as nn
import numpy as np
import pandas as pd
from typing import Optional, Tuple
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

def generate(
        model: nn.Module, 
        tokenizer: PreTrainedTokenizerBase, 
        start: str, 
        max_length: int = 100, 
        eps: float = 0.5, 
        top_n: int = 5,
        return_iterations: bool = False,
        device: str = "cpu") -> Tuple[str, Optional[pd.DataFrame]]:

    output = [start]
    iterations = []
    with torch.no_grad():
        input_ids = tokenizer(output[-1], return_tensors='pt')['input_ids'].to(device)
        for _ in range(max_length):
            logits = model(input_ids=input_ids).logits
            probs = torch.softmax(logits[0, -1, :], dim=-1)
            sorted_tokens = torch.argsort(probs, dim=-1, descending=True)
            if np.random.random_sample(1)[0] < eps:
                # Se toma el mejor token
                next_token = sorted_tokens[0].unsqueeze(dim=0)
            else:
                # Se muetrea el token de la probabilidad de distribución
                next_token = torch.multinomial(probs, 1)
            
            if return_iterations:
                # Mantenemos pista de todas las iteraciones para análisis
                iteration = {'input': ''.join(output)}
                best_n = sorted_tokens[:top_n].cpu().tolist()
                choices = {f'Choice #{choice+1}': f'{tokenizer.decode(token)} ({prob:.4f})' for choice, (token, prob) in enumerate(zip(best_n, probs[best_n].cpu().tolist()))}
                iteration.update(choices)
                iterations.append(iteration)

            output.append(tokenizer.decode(next_token))
            input_ids = torch.cat([input_ids, next_token.unsqueeze(dim=0)], dim=-1)

        output_text = ''.join(output)
        if not return_iterations:
            return output_text, None
        else:
            df = pd.DataFrame(iterations)
            return output_text, df

Primero, observemos que pasa cuando pasamos un `eps=1` que quiere decir que la generación va a ser de tipo greedy:

In [7]:
output_text, iterations_df = generate(model, tokenizer, text, max_length=15, eps=1.0, top_n=10, return_iterations=True, device=device)
print(output_text)
iterations_df.head(15)

Quiero algo rico con pollo.¿Qué?¿Qué?¿Qué?¿Qué?¿Qué


,input,Choice #1,Choice #2,Choice #3,Choice #4,Choice #5,Choice #6,Choice #7,Choice #8,Choice #9,Choice #10
0,Quiero algo rico con pollo,. (0.3899),y (0.1726),", (0.1124)",frito (0.0453),... (0.0322),para (0.0300),a (0.0190),en (0.0157),con (0.0089),! (0.0080)
1,Quiero algo rico con pollo.,¿ (0.1479),- (0.0945),No (0.0465),¡ (0.0454),Y (0.0348),Quiero (0.0157),Me (0.0140),Un (0.0138),Lo (0.0137),Sí (0.0131)
2,Quiero algo rico con pollo.¿,Qué (0.1838),Y (0.0484),Quieres (0.0473),Por (0.0352),No (0.0271),Dónde (0.0252),Cómo (0.0226),Te (0.0222),Sabes (0.0209),Tienes (0.0200)
3,Quiero algo rico con pollo.¿Qué,? (0.1499),tal (0.1057),te (0.0836),es (0.0812),quieres (0.0575),le (0.0283),hay (0.0245),estás (0.0191),pasa (0.0190),tienes (0.0175)
4,Quiero algo rico con pollo.¿Qué?,¿ (0.2757),No (0.0716),¡ (0.0563),- (0.0545),Nada (0.0223),Algo (0.0181),Un (0.0177),Quiero (0.0167),Me (0.0111),Es (0.0111)
5,Quiero algo rico con pollo.¿Qué?¿,Qué (0.2289),No (0.0450),Quieres (0.0449),Por (0.0404),Un (0.0265),De (0.0205),Cómo (0.0197),Tienes (0.0191),Pol (0.0172),Te (0.0164)
6,Quiero algo rico con pollo.¿Qué?¿Qué,? (0.2414),quieres (0.1340),es (0.1171),tal (0.0327),estás (0.0323),te (0.0318),pasa (0.0318),", (0.0209)",tienes (0.0200),le (0.0133)
7,Quiero algo rico con pollo.¿Qué?¿Qué?,¿ (0.3148),No (0.0720),- (0.0666),¡ (0.0658),Nada (0.0192),Es (0.0117),Me (0.0105),Tengo (0.0104),Un (0.0099),Quiero (0.0098)
8,Quiero algo rico con pollo.¿Qué?¿Qué?¿,Qué (0.4090),Quieres (0.0415),No (0.0351),Por (0.0346),De (0.0255),Cómo (0.0183),Te (0.0158),Un (0.0151),Estás (0.0139),Tienes (0.0130)
9,Quiero algo rico con pollo.¿Qué?¿Qué?¿Qué,? (0.3171),quieres (0.1444),es (0.1127),estás (0.0340),pasa (0.0310),te (0.0231),", (0.0194)",tienes (0.0183),quiere (0.0138),está (0.0133)


Observamos como el input progresa a la vez que las opciones de tokens que hay. Sin importar cuantas veces invoquemos a la función con los mismos parámetros, siempre vamos a obtener los mismos resultados, en este caso como es algo es muy especializado, el modelo general no muestra buenos resultados, si se baja la eps, va ha ser peor el resultado

In [8]:
output_text, _ = generate(model, tokenizer, text, max_length=100, eps=0.5, device=device)
print(output_text)

Quiero algo rico con pollo.Perfecto.Lo que sea.Está bien.¿Quieres algo?¿Café?¿Té?¿Café?¿Hablan los blancos?Si supiste blanco, bien.Me sorprende que sepa inglés.Porque los chinos no hablan muy bien.Tampoco entienden el mandarín.¿Cómo dices que no hablas bien el chino?¿No puede hablar otra persona?¿Qué es esto?¿Es un mercado?Este es nuestro perro.No es un mercado.Quizá quiere hablarte.No sonrías.¿


### Generando texto con las utilidades del modelo

In [9]:
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.5, top_k=0)
print(tokenizer.decode(output[0]))

Quiero algo rico con pollo.¿Qué?¿Qué es esto?¿Qué es esto?¿Qué es esto?¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¿Qué es esto?¿Qué es esto?¿Qué es esto?¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡Oye!¡


## Fine tuning

Ahora, intentemos hacer fine tuning a nuestro modelo. Intentemos entrenarlo en un recetas de cocina escogiendo idioma español y ver como la narrativa de su output cambia.

In [10]:
from datasets import load_dataset

dataset = load_dataset("Frorozcol/recetas-cocina")
dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'url', 'ingredients', 'steps', 'uuid'],
        num_rows: 20526
    })
    validation: Dataset({
        features: ['title', 'url', 'ingredients', 'steps', 'uuid'],
        num_rows: 2781
    })
    test: Dataset({
        features: ['title', 'url', 'ingredients', 'steps', 'uuid'],
        num_rows: 3899
    })
})

In [11]:
dataset['train'][0]

{'title': 'Gratinado de puerros y langostinos',
 'url': 'https://www.recetas.com/receta-de-Gratinado-de-puerros-y-langostinos-4220.html',
 'ingredients': '400 gr. de langostinos o camaronesMantecaSal2 cdas. e yogur natural50 gr. de queso rallado200 gr. de crema de leche 3 yemas de huevo 8 puerros ',
 'steps': ' Gratinado de puerros y langostinos Preparación: Pelar los langostinos o camarones y sofreír a fuego suave, en manteca. Salpimentar y reservar. Lavar y cortar la parte blanca de los puerros en rodajas finas y freír en manteca unos minutos, hasta que empiecen a tomar color. Agregar la crema de leche y el yogur, y cocer todo junto unos minutos más. Retirar del fuego, añadir las yemas batidas y las gambas reservadas, mezclar bien. Rectificar el sazonado. Verter en cazuelitas enmantecadas, espolvorear con queso rallado y gratinar. Servir inmediatamente. ',
 'uuid': '4319f673-bf9b-4228-b7d6-003944883554'}

In [12]:
from datasets import Dataset
import pandas as pd

dataset.set_format('pandas')
df = dataset['train'].to_pandas()
df.head(10)

,title,url,ingredients,steps,uuid
0,Gratinado de puerros y langostinos,https://www.recetas.com/receta-de-Gratinado-de...,400 gr. de langostinos o camaronesMantecaSal2 ...,Gratinado de puerros y langostinos Preparació...,4319f673-bf9b-4228-b7d6-003944883554
1,Sorbete de cava,https://www.recetas.com/receta-de-Sorbete-de-c...,Ingredientes 2/3 taza de azúcar 1/2 taza de ag...,Sorbete de cava Ponga el azúcar y el agua en ...,303f3c25-29a3-46c4-be77-9d02941e8f19
2,Ceviche peruano de Conchas negras,https://www.recetasgratis.net/receta-de-cevich...,2 docenas de conchas negras 1 cebolla roja pic...,1 Abrir las conchas negras cuidadosamente. 2 R...,b1e388c2-b831-40b0-b893-e6aef268a04f
3,Frozen yogurt,https://www.recetas.com/frozen-yogurt-ld8.html,500 gr. de yogur blanco 150 ml de leche 80 gr....,Frozen yogurt El yogurt es un postre dulce si...,9278bfa1-2132-43ef-b819-3a96c8388949
4,Tortitas de arroz integral caseras,https://www.recetasgratis.net/receta-de-tortit...,10 unidades de Champiñones 1 unidad de Cebolla...,1 En un bol pequeño pon la harina de garbanzos...,765b6fae-bb85-49da-bb85-8e664ed50611
5,Chicharrones de Guayaba,https://www.mycolombianrecipes.com/es/chicharr...,"12 chicharrones 2 hojas 1 libra de hojaldre, d...",Espolvorear una superficie de trabajo con hari...,b4f4ed10-f309-4044-a3cd-f3199ac99457
6,Flan de vainilla y caramelo,https://www.recetasgratis.net/receta-de-flan-d...,1 lata leche evaporada 1 lata leche condensada...,1 Mezcla todos los ingredientes en la licuador...,df5f6248-e0df-45f0-a5be-2db0300d2e7c
7,Perros Calientes Colombianos,https://www.mycolombianrecipes.com/es/perros-c...,2 tazas de piña fresca pelada y cortada en tro...,Para hacer la salsa: coloque la piña y el agua...,04eaa2bd-d280-4d20-8f71-e40a793ee69b
8,Pasta arrabiata con camarones,https://www.recetasnestle.com.co/recetas/pasta...,1 cucharada aceite (14 g) 1/2 libra de...,1. En una sartén calienta el aceite a fue...,78f97b23-ffab-4252-8f7c-033fad509ede
9,Dulce de leche repostero,https://www.recetasgratis.net/receta-de-dulce-...,1 litro de leche entera 300 gramos de azucar (...,1 Coloca en una cacerola la leche entera y el ...,e790e440-27f9-47f2-a21a-d6615306d01b


In [13]:
df['title'] = df['title'].str.split().str.len()
df['title'].median()

np.float64(4.0)

In [14]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        texts = [t if t is not None else "" for t in examples['title']]
        return tokenizer(texts, max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function

Los modelos GPT no esperan otra cosa más que los `input_ids`, por lo que retirarémos todas las demás columnas del dataset ya que no nos son de utilidad en este momento. 

In [15]:
dataset.reset_format()
dataset_es = dataset['train']
tokenized_dataset = dataset_es.map(preprocess_function(max_len=512), batched=True)
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col != 'input_ids'])
tokenized_dataset = tokenized_dataset.train_test_split(train_size=0.9, seed=42)
tokenized_dataset.set_format('torch')
tokenized_dataset

Map:   0%|          | 0/20526 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 18473
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 2053
    })
})

Finalmente procedemos a definir el entrenamiento. Observaremos que es muy similar a como entrenamos a BERT.

In [16]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments


batch_size = 8 if IN_COLAB else 2
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf-gpt',
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='none'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
)

In [17]:
%%time
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.538632,2.357083
2,2.080064,2.286749
3,1.854890,2.297266
4,1.671005,2.373779
5,1.507755,2.494035
6,1.362882,2.620791
7,1.234400,2.771704
8,1.131557,2.902029
9,1.049132,3.008373
10,0.984690,3.125524


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


CPU times: total: 1h 47min 32s
Wall time: 1h 48min 52s


TrainOutput(global_step=92370, training_loss=1.5414378087835754, metrics={'train_runtime': 6532.0025, 'train_samples_per_second': 28.281, 'train_steps_per_second': 14.141, 'total_flos': 4.826847707136e+16, 'train_loss': 1.5414378087835754, 'epoch': 10.0})

Ahora observemos los resultados.

In [18]:
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.8)
print(tokenizer.decode(output[0]))

Quiero algo rico con pollo y mantequilla de cacahuete.¿Cómo va hacer para hacer paracetamol?Picarón de lenguado con setas y queso parmesano.¿Cómo hacer queso parmesano?A ver ¿en qué se diferencia el queso parmesano de estofado de ternera?A ver ¿en qué se diferencia el queso parmesano?A ver ¿en qué se diferencia el queso parmesano?A ver ¿en qué se diferencia el queso par


In [19]:
output_text, _ = generate(model, tokenizer, text, max_length=100, eps=0.2, device=device)
print(output_text)

Quiero algo rico con pollo a la napolitana y champiñones cookies.- ¿Champiñones de marisco?- ¡Ja!Muy fácil y crujiente.Perfecto para fiestas infantiles y adultos.- ¡De chocolate a la napolitana!- ¡Muy fácil y crujiente!¡Muy fácil y delicioso y delicioso!Perfecto para fiestas infantiles y adultos.- ¡Muy fácil y crujiente!- ¡Muy fácil y delicioso!¡Muy fácil y sabroso!¡Muy fácil y delicioso!- ¡Muy


## Descripción de resultados

En este notebook se trabajó con un modelo GPT-2 en español en dos etapas: (1) generación con el modelo base pre-entrenado y (2) fine-tuning sobre el dataset de recetas de cocina.

El entrenamiento se ejecutó con una configuración (10 épocas, `eval_strategy='no'`, `save_strategy='no'`, `fp16=True`) para reducir tiempo de cómputo y permitir iteraciones más ágiles.

## Conclusiones

- El pipeline completo se ejecutó correctamente: carga de datos, limpieza, tokenización, entrenamiento y generación.
- Filtrar a español antes del fine-tuning mejora la coherencia lingüística del ajuste y reduce ruido de ejemplos en otros idiomas.
- El modelo ajustado mostró cambio de estilo frente al modelo base, con salida más orientada al dominio del corpus usado para entrenamiento.
- En modelos generativos, la calidad final depende tanto de los hiperparámetros como de la curación del dataset; en este caso, el filtrado por idioma fue una mejora clave.
- Para iteración rápida, la configuración reducida de entrenamiento fue efectiva; como siguiente paso, conviene evaluar una corrida más larga con validación activa para comparar calidad de forma más rigurosa.